# Lecture 14 - Fourier Transforms

Many fields of astrophysics --- Stellar astrophysics, planetary astrophysics, exoplanetary astrophysics, high-energy astrophysics, galactic astrophysics, etc. --- need to search for periodic signals in lightcurves. The way this is typically done is to convert the lightcurve from being a function of *time* $f(t)$, to being a function of *frequency* $F(\nu)$. When $F(\nu)$ is large near a particular frequency $\nu$, the signal likely has some interesting astrophysics causing the signal to be periodic near $\nu$. This lecture discusses the primary techniques one transforms $f(t) \to F(\nu)$.

### Accepting and submitting

To accept this assignment on Classroom 50 with your GitHub profile enrolled in the class, click on the link \
https://classroom50.org/PsuAstro410/410astro26/assignments/lecture-14/accept \
and follow the steps the webpage prompts.

To submit this assignment, follow the steps in "First time Classroom 50 setup" and "Accepting and Submitting Assignments" on the course webpage:\
https://psuastro410.github.io/tips/github/

### Resources and Acknowledgements

We use material from the following sources:
https://zingale.github.io/computational_astrophysics/ffts/fourier.html \
https://zingale.github.io/computational_astrophysics/ffts/dft.html \
https://zingale.github.io/computational_astrophysics/ffts/dft-tests.html \
https://zingale.github.io/computational_astrophysics/ffts/fft.html \
https://zingale.github.io/computational_astrophysics/ffts/application-lightcurves.html

In [ ]:
## Nice things to have
import numpy as np
import matplotlib.pyplot as plt

# Fourier Transforms

The [Fourier transform](https://en.wikipedia.org/wiki/Fourier_transform) converts a function from a physical-space (or time) representation
into a frequency-space representation.  The two representations are equivalent, but give
alternate views into how the function behaves.

The general form of a Fourier transform is:
$$F(k) = \int_{-\infty}^{\infty} f(x) e^{-2\pi i k x} dx$$
where $f(x)$ is the real-space form of the function. If we think of this in terms of space, $x$ has dimensions of length and $k$ has dimensions of 1/length, and can be interpreted as a wavenumber. You can think of $F(k)$ as being the amount of the function $f$ represented by a frequency $k$.

If we think of it in terms of time, then $f(t)$ is the time-domain function and the transform is:
$$F(\nu) = \int_{-\infty}^{\infty} f(t) e^{-2\pi i \nu t} dt$$
and $\nu$ is now a frequency (units of 1/time).

The inverse transform is then:
$$f(x) = \int_{-\infty}^\infty F(k) e^{2\pi ikx} dk$$

## Discrete Fourier transform

The discrete Fourier transform (DFT) works on discrete data, e.g.:
* a time-series from an experiment
* simulation data for a velocity field
  
**Note:** We'll work on the assumption that our data is evenly-spaced.

The discrete analogue of the Fourier transform is:
$$F_k = \sum_{n = 0}^{N-1} f_n e^{-2\pi i n k / N}$$
where we have $N$ samples with values $f_n$. The inverse transform is:
$$f_n = \frac{1}{N} \sum_{k=0}^{N-1} F_k e^{2\pi i n k / N}$$

**Note:** The convention we choose here is the same as used by the NumPy library in [numpy.fft](https://numpy.org/doc/stable/reference/routines.fft.html)

Sometimes we write the DFT operation as:
$$F_k = \mathcal{F}(f_n)$$
and the inverse as:
$$f_n = \mathcal{F}^{-1}(F_k)$$

### Significance of real and imaginary parts

Recall that we are integrating with a factor $e^{-2\pi i k x}$.  Euler's formula
tells us that

$$e^{ix} = \cos(x) + i \sin(x)$$

This means that the real part of the transform represents the cosine terms (symmetric functions) and the imaginary part represents the sine terms (antisymmetric).

If $f_n$ is real, then:

$$\mathrm{Re}(F_k) = \sum_{n=0}^{N-1} f_n \cos \left ( \frac{2 \pi n k}{N} \right )$$
$$\mathrm{Im}(F_k) = \sum_{n=0}^{N-1} f_n \sin \left ( \frac{2 \pi n k}{N} \right )$$


**Note:**
$k = 0$ is special:
$$\mathrm{Re}(F_0) = \sum_{n=0}^{N-1} f_n \cos \left ( \frac{2 \pi n (0)}{N} \right ) = \sum_{n=0}^{N-1} f_n$$
$$\mathrm{Im}(F_0) = \sum_{n=0}^{N-1} f_n \sin \left ( \frac{2 \pi n (0)}{N} \right ) = 0$$
This is sometimes called the _DC offset_.


### Normalization

We haven't yet motivated the $1/N$ in front of the inverse DFT.  This can be shown to arise
from the discrete form of the [Plancherel theorem](https://en.wikipedia.org/wiki/Plancherel_theorem)&mdash;namely that both the real-space
and frequency-space form of the function contain the same power.

But we can motivate it by a simple example.

Consider $f(x) = 1$. If we look at the discrete transform then:

$$F_k = \sum_{n=0}^{N-1} 1 \cdot e^{-2\pi i k n /N}
= \sum_{n=0}^{N-1} \left [ \cos \left (\frac{2\pi kn}{N}\right) - i \sin \left (\frac{2\pi kn}{N}\right ) \right ]$$

* For $k = 0$: $\cos(0) = 1$ and $\sin(0) = 0$, so $F_0 = N$
* For $k > 0$: we are essentially doing a sum over the cosine and sine using equally spaced points, and the sum is always over a multiple of a full wavelength $\rightarrow$ $F_k = 0$

With these frequency-space $F_k$'s, we can compute the inverse transform:

$$f_n = \frac{1}{N} \sum_{k=0}^{N-1} F_k e^{2\pi i nk/N}
 = \frac{1}{N} N e^{2\pi i n 0/N} = 1$$

So we see that the $1/N$ is needed to get back the original value of $f(x)$.

### Continuum limit

When there are infinitely many points ($N \to \infty$), the Fourier transform gives
$$ f(x) = 1 \Longleftrightarrow F(k) = \delta(k)$$
Here, $\delta(k)$ is called the *Dirac delta function*, which has values of
$$ \delta(x) = 0 \ \text{when} \ x \ne 0 $$
$$ \delta(x) = \infty \ \text{when} \ x = 0 $$
and has the property that its integral
$$ \int_{-\infty}^{\infty} \delta(x) dx = 1. $$

# Discrete Fourier Transform Code

Let's implement a discrete Fourier transform.

We essentially want to code up:

$$F_k = \sum_{n = 0}^{N-1} f_n e^{-2\pi i n k / N}$$

This requires a double sum: for each wavenumber, $k$, we sum over all the spatial points. This means that the amount of work scales as $\mathcal{O}(N^2)$. Later on, we will discuss how to save computational time on this algorithm.

In [ ]:
def dft(f_n):
    """perform a discrete Fourier transform"""
    
    N = len(f_n)
    n = np.arange(N)
    
    f_k = np.zeros((N), dtype=np.complex128)

    for k in range(N):
        f_k[k] = np.sum(f_n * np.exp(-2.0 * np.pi * 1j * n * k / N))
    return f_k

**Note:**
In python, the complex unit, $\sqrt{-1}$ is denoted by `j` immediately following the number.

Now let's try this out.  Let's compute the transform of 
$$f(x) = \sin(2 \pi \nu_0 x)$$
We would expect to see all of the power at a single frequency $\nu_0$, and since our function is a $\sin()$, we would expect for all of the power to be in the imaginary component.

In [ ]:
N = 64
nu_0 = 0.2
xmax = 50.0

x = np.linspace(0.0, xmax, N, endpoint=False)
f_n = np.sin(2.0 * np.pi * nu_0 * x)

fig, ax = plt.subplots()
ax.plot(x, f_n)
plt.show()

Now the DFT:

In [ ]:
f_k = dft(f_n)
nu_k = np.arange(N) / xmax

fig, ax = plt.subplots()
ax.plot(nu_k, f_k.real, label=r"Re($F_k$)")
ax.plot(nu_k, f_k.imag, ls=":", label=r"Im($F_k$)")

ax.set_xlabel(r"$\nu_k$")
ax.set_ylabel(r"$F_k$")
ax.legend()
plt.show()

As expected, all of the power is in the imaginary part, and it looks like it is at a frequency of $\nu = 0.2$, as we expected.  But there are 2 peaks.

We now need to discuss this following:

* How did we compute the physical frequency, $\nu_k$, corresponding to each frequency sample?
* Why are there 2 peaks in our transform?
* What is the significance of the amplitudes in this plot?

## Frequency

We want to understand how to get a physical frequency from the data. Consider our single-mode sine wave.  We can express it as:

$$\sin(2\pi \nu x) =  \sin \left ( \frac{2\pi k x}{L} \right ) = \sin \left ( \frac{2\pi k n}{N} \right )$$

where $L$ is the maximum extent of the coordinate $x$ (if $x$ represents time, then this would be the maximum time for the data), and $n$ is the index of the discrete data. They are related via

$$x_n = \frac{n L}{N}$$

![sine waves](https://zingale.github.io/computational_astrophysics/_images/sine-k.png)

From this plot, we see that if there is a single mode in the domain, associated with the wavenumber `1`, then the physical frequency is $1/L$.

And in general, $\nu = k/L$ for $k = 0, 1, 2, \ldots N-1$.

Most DFT routines will return frequencies in index space, e.g.,

$$k_\mathrm{freq} = 0, 1/N, 2/N, 3/N, \ldots$$

In this index space, the shortest wavelength is from one cell to the next, and the corresponding frequency is $1/\Delta x$

The physical scale for the frequency is then:

$$\nu_k = \underbrace{\left ( \frac{k}{N} \right )}_{k_\mathrm{freq}} \cdot \frac{1}{\Delta x} = \left (\frac{k}{N} \right ) \cdot \frac{N}{L} = \frac{k}{L}$$

* lowest frequency: $1/L$, corresponds to the largest wavelength, e.g., the entire domain
* highest frequency: $N/L \sim 1/\Delta x$, corresponds to the shortest wavelength, one sample to the next

Following this procedure, the physical frequency for our DFT above is found simply via:
    
```python
nu_k = np.arange(N) / xmax
```

### Nyquist frequency

The highest meaningful frequnecy is $1/(2 \Delta x)$ which is called the [Nyquist frequency](https://en.wikipedia.org/wiki/Nyquist_frequency).

Consider a timeseries with 10 samples that goes for 10 s.  So $\Delta t = 1 s$.  The Nyquist frequency would be $\nu_\mbox{Nyquist} = 1/(2 \Delta t) = 0.5~\mbox{Hz}$.

Using our prescription above, the frequencies are:

$$\nu_k = 0, \frac{1}{10~\mbox{s}}, \frac{2}{10~\mbox{s}}, \frac{3}{10~\mbox{s}}, \frac{4}{10~\mbox{s}}, {\color{red}\frac{5}{10~\mbox{s}}}, \frac{6}{10~\mbox{s}}, \frac{7}{10~\mbox{s}}, \frac{8}{10~\mbox{s}}, \frac{9}{10~\mbox{s}}$$

where we highlight the Nyquist frequency in red.

These are equivalent to

$$\nu_k = 0, 0.1~\mbox{Hz}, 0.2~\mbox{Hz}, 0.3~\mbox{Hz}, 0.4~\mbox{Hz}, {\color{red} 0.5~\mbox{Hz}}, 0.6~\mbox{Hz}, 0.7~\mbox{Hz}, 0.8~\mbox{Hz}, 0.9~\mbox{Hz}$$

Let's look at what happens when we try to represent a sine wave with 10 points at these different frequencies:

In [ ]:
def f_sin(t, nu):
    return np.sin(2 * np.pi * nu * t)

fig, axs = plt.subplots(nrows=5, ncols=2, layout="constrained", figsize=(8, 12))

t = np.linspace(0, 10, 10)
tt = np.linspace(0, 10, 1000)

for n, ax in enumerate(axs.flat):
    nu = 0.1 * n
    ax.plot(t, f_sin(t, nu), label=f"nu = {nu:4.2f} s")
    ax.plot(t, f_sin(t, nu), ls='', marker='o', color='k')
    ax.plot(tt, f_sin(tt, nu), linestyle=":")
    ax.legend(fontsize="small")
    ax.set_xlabel="time (s)"


We see that once we go past the Nyquist frequency, we no longer resolve the correct number of wave peaks.  This is an example of [_aliasing_](https://en.wikipedia.org/wiki/Aliasing)

## Why 2 peaks?

Both the real-space and frequency-space forms of the function contain the same information.  In our `dft()` function above, we had N real values and return N complex values,
but each complex number has twice the amount of information as a single real number.  This means that half of the complex numbers we return are not independent.

This is a consequence of the fact that for an analytic Fourier transform of a real function, $f(x)$, $F(-k) = F^*(k)$.  In the discrete transform, this shows itself as:

$$F_{N-k} = F_k^\star$$

we can see this as:

$$F_{N-k} = \sum_{n=0}^{N-1} f_n e^{-2\pi i n (N-k)/N} = \sum_{n=0}^{N-1} f_n e^{+2\pi i n k/N} \underbrace{e^{-2\pi i n}}_{= 1} = F_k^\star$$

So only the first $N/2$ points we get from our DFT routine are important.  The others don't add any new information.  This will also affect the normalization (as discussed below).

The also means that the maximum frequency is $1/(2\Delta x)$.

## How do we interpret the amplitude?

When plotting, we want to put the physical scale back in, as well as make the correspondence to the continuous representation.

We can rewrite the inverse DFT expression as:

$$f_n = \frac{1}{N} \sum_{n=0}^{N-1} F_k e^{2\pi i n k /N} 
= \underbrace{\sum_{n=0}^{N-1} \overbrace{\left (\frac{F_k}{N} \right )}^{\substack{\mbox{this is what}\\ \mbox{we plot}}}
e^{2\pi i n k /N}}_{\mbox{this looks like the continuous form}} $$

* This suggests that we should normalize the $F_k$ by the number of points, $N$.
* Another interpretation relates to the continuous form of the Fourier transform: $F_k / N \Longleftrightarrow F_k dk$

## Power spectrum

The power spectrum is defined simply as

$$P(k) = |F(k)|^2 = F(k) F^\star(k)$$

This is a single number that shows the importance / weight of a given frequency

# Testing our DFT

A good thing to check is that if we compute the DFT and then the inverse DFT of the result, do we get the original data back (up to roundoff error)? First let's copy over our `dft()` function and write the inverse transform:

In [ ]:
def dft(f_n):
    """perform a discrete Fourier transform"""
    
    N = len(f_n)
    n = np.arange(N)
    
    f_k = np.zeros((N), dtype=np.complex128)

    for k in range(N):
        f_k[k] = np.sum(f_n * np.exp(-2.0 * np.pi * 1j * n * k / N))
    return f_k

def idft(F_k):
    """perform an inverse discrete Fourier transform"""
    
    N = len(F_k)
    k = np.arange(N)
    
    f_n = np.zeros((N), dtype=np.float64)

    for n in range(N):
        f_n[n] = (1/N) * np.sum(F_k * np.exp(2.0 * np.pi * 1j * n * k / N)).real
    return f_n

Now let's make a function that plots the original function, the DFT, the power spectrum, and the inverse transform of the DFT

In [ ]:
def plot_dft(f, N=64, xmax=50):
    
    fig, ax = plt.subplots(nrows=4)
    
    x = np.linspace(0.0, xmax, N, endpoint=False)
    
    # get the transform   
    f_n = f(x)
    F_k_raw = dft(f_n)
    
    # normalize and cut off the duplicate data
    # since we are discarding 1/2 the data, the normalization if N/2
    F_k = F_k_raw[0:N//2] * 2 / N
    
    # compute the frequencies
    nu_k = np.arange(len(F_k)) / xmax
    
    # plot the original function
    ax[0].plot(x, f_n)
    ax[0].set_xlabel("$x$")
    ax[0].set_ylabel("$f(x)$")
    ax[0].set_xmargin(0.0)
    
    # plot the transform
    ax[1].plot(nu_k, F_k.real, label=r"$\mathrm{Re}(F_k)$")
    ax[1].plot(nu_k, F_k.imag, label=r"$\mathrm{Im}(F_k)$")    
    ax[1].set_xlabel(r"$\nu$")
    ax[1].set_ylabel("$F_k$")
    ax[1].set_xmargin(0.0)
    ax[1].legend(fontsize="small", ncol=2, frameon=False)
    
    # plot the power spectrum
    ax[2].plot(nu_k, np.abs(F_k)**2)
    ax[2].set_xlabel(r"$\nu$")
    ax[2].set_ylabel("$|F_k|^2$")
    ax[2].set_xmargin(0.0)
    
    # take the inverse transform -- note we need to use the unnormalized data
    f_n_new = idft(F_k_raw)
    ax[3].plot(x, f_n_new)
    ax[3].set_xlabel("$x$")
    ax[3].set_ylabel("$f(x)$")
    ax[3].set_xmargin(0.0)
    
    fig.tight_layout()
    
    return fig

## Transform of sin(x)

Now let's look at the transform of a single mode sine

In [ ]:
def f_single_sine(x):
    nu_0 = 0.2
    return np.sin(2.0 * np.pi * nu_0 * x)

fig = plot_dft(f_single_sine, N=256)

We get what we expect:

* The transform only has non-zero weights in the imaginary part
* All of the power is at the single frequency that we constructed the sine with
* The inverse transform of the transformed data gives us back our original function

**Note:**
The Fourier transform treats the data as if it is periodic (since the sine and cosine functions are).  The input data is assumed to be equally spaced and periodic.  The means that the first and last point need to be distinct.

The assumption of periodicity is why we construct the `x` data with `endpoint=False`&mdash;if we have data at endpoint, then the start and end refer to the exact same data sample, but the Fourier transform will treat them as if they are space $\Delta x$ apart, and we will pick up an extra signal.

## Transform of cos(x)

Now let's look at cosine&mdash;it is the opposite of sine

In [ ]:
def f_single_cosine(x):
    nu_0 = 0.2
    return np.cos(2.0 * np.pi * nu_0 * x)

fig = plot_dft(f_single_cosine, N=256)

## Introducing a phase

A $\pi/4$ phase shift puts equal weight into the sine and cosine components:

In [ ]:
def f_single_sine_phase(x):
    nu_0 = 0.2
    return np.sin(2.0 * np.pi * nu_0 * x + np.pi / 4)

fig = plot_dft(f_single_sine_phase, N=256)

## Multiple frequencies

The two frequency version shows power at the two frequencies, as expect:

In [ ]:
def f_two_freq(x):
    nu_0 = 0.2
    nu_1 = 0.5
    return 0.5 * (np.sin(2.0 * np.pi * nu_0 * x) +
                  np.sin(2.0 * np.pi * nu_1 * x))

fig = plot_dft(f_two_freq, N=256)

# Fast Fourier Transform

So far we have been working with the _discrete Fourier transform_ (DFT).  This does what we want, but its complexity scales like $\mathcal{O}(N^2)$, so for really large datasets, it will be slow. *The fast Fourier transform (FFT) is algebraically identical to the DFT*, but uses some clever tricks to speed up the computation, resulting in an algorithm that scales like $\mathcal{O}(N\log N)$.

The standard algorithm for the FFT is the [Cooley-Tukey FFT](https://en.wikipedia.org/wiki/Cooley%E2%80%93Tukey_FFT_algorithm).  Not only does it gain speed from a divide-and-conquer algorithm, it also does the transform _in-place_, saving memory.  

The FFT can be written as:
    
\begin{align*}
F_k &= \sum_{n=0}^{N-1} f_n e^{-2\pi i k n /N} \\
 &=   \sum_{r=0}^{N/2-1} f_{2r} e^{-2\pi i k (2r) / N} +
   \sum_{r=0}^{N/2-1} f_{2r+1} e^{-2\pi i k (2r+1)/N}
\end{align*}

where we simply split the even and odd terms.

Now look at the even terms:

$$E_k \equiv \sum_{r=0}^{N/2-1} f_{2r} e^{-2\pi i k (2r)/N} =
   \sum_{r=0}^{N/2-1} f_{2r} e^{-2\pi i k r / (N/2)}$$
   
This is just the DFT of the $N/2$ even samples.

The odd terms are:

\begin{align*}
\sum_{r=0}^{N/2-1} f_{2r+1} e^{-2\pi i k (2r+1)/N} &=
 e^{-2\pi i k/N}   \sum_{r=0}^{N/2-1} f_{2r+1} e^{-2\pi i k r / (N/2)} \\
&\equiv e^{-2\pi i k/N} O_k
\end{align*}

Here, $O_k$ is just the DFT of the $N/2$ odd samples

If we define:
    
$$\omega^k \equiv e^{-2\pi i k /N}$$

then we can write our original transform as:
    
$$F_k = E_k + \omega^k O_k$$

In doing this, we went from one DFT involving $N$ samples to 2 DFTs with $N/2$ samples.  The number of wavenumbers also is cut in half.

Periodicity tells us that:

$$F_{k+N/2} = E_k - \omega^k O_k$$

since $E_k$ and $O_k$ are periodic with $N/2$

Let's consider 8 samples:

$$f_n = \{ f_0, f_1, f_2, f_3, f_4, f_5, f_6, f_7 \}$$

* Express this in terms of the FFTs of the even and odd terms:

  $$F_k = \mathcal{F}(f_0, f_2, f_4, f_6) + \omega^k \mathcal{F}(f_1, f_3, f_5, f_7)$$
  
  this is defined for $k = 0, 1, 2, 3$, since that's what each of the $N/2$ samples gets.
  
* The 2nd half of the frequencies are:

  $$F_{k+N/2} = \mathcal{F}(f_0, f_2, f_4, f_6) - \omega^k \mathcal{F}(f_1, f_3, f_5, f_7)$$
  
  This gives $k = 4, 5, 6, 7$
  
So 2 FFTs of 4 samples each gives us the FFT of 8 samples defined over 8 wavenumbers.

We then apply this recursively.

We eventually get down to $N$ FFTs of 1 sample each:

$$\mathcal{F}(f_0) = f_0$$

**Note:**
The procedure that we outlined above (and implement below) assumes that the number of data points is a power of 2.

## Implementation

In [ ]:
def fft(f_n):
    """ perform a discrete Fourier transform.  We use the same
        conventions as NumPy's FFT

        here,

              N-1         -2 pi i n k /N
        F_k = sum  f   exp
              i=0   n

    """

    N = len(f_n)

    if N == 1:
        return f_n
    else:

        # split into even and odd and find the FFTs of each half
        f_even = f_n[0:N:2]
        f_odd = f_n[1:N:2]

        F_even = fft(f_even)
        F_odd = fft(f_odd)

        # combine them.  Each half has N/2 wavenumbers, but due to
        # periodicity, we can compute N wavenumbers
        omega = np.exp(-2*np.pi*1j/N)

        # allocate space for the frequency components -- they are,
        # in general, complex
        F_k = np.zeros((N), dtype=np.complex128)

        oterm = omega**np.arange(N//2)
        F_k[0:N//2] = F_even + oterm * F_odd
        F_k[N//2:] = F_even - oterm * F_odd

    return F_k

We'll also copy over our DFT:

In [ ]:
def dft(f_n):
    """perform a discrete Fourier transform"""
    
    N = len(f_n)
    n = np.arange(N)
    
    f_k = np.zeros((N), dtype=np.complex128)

    for k in range(N):
        f_k[k] = np.sum(f_n * np.exp(-2.0 * np.pi * 1j * n * k / N))
    return f_k

## Testing

Let's create some random data and compare our FFT against our DFT.

In [ ]:
def data(xmin=0.0, xmax=50.0, npts=32):
    """create a multi-frequency, multi-phase sine dataset"""
    
    xx = np.linspace(0.0, xmax, npts, endpoint=False)

    # number of frequencies to superpose
    nfreq = 48

    # frequency in terms of 1/length
    freqs = np.random.randint(1, npts, nfreq) / xmax
    phases = np.random.uniform(0.0, 2.0*np.pi, nfreq)
    
    f_n = np.zeros_like(xx)
    for f, p in zip(freqs, phases):
        f_n += np.sin(2.0*np.pi*f*xx + p)

    return xx, f_n

xmax = 50.0
xx, f_n = data(xmax=xmax, npts=512)

fig, ax = plt.subplots()

ax.plot(xx, f_n)
plt.show()

Let's ensure they get the same answer

In [ ]:
f_k_dft = dft(f_n)
f_k_fft = fft(f_n)

N = len(f_k_dft)
k = np.arange(N) / xmax

fig, ax = plt.subplots()

ax.plot(k[:N//2], np.abs(f_k_dft[:N//2]))
ax.plot(k[:N//2], np.abs(f_k_fft[:N//2]))

ax.set_xlabel(r"$\nu_k$")
ax.set_ylabel(r"$|F_k|$")

plt.show()

## Performance

Now let's time them

In [ ]:
import time

Ns = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384]

dft_times = []
fft_times = []

for N in Ns:
    xx, f_n = data(npts=N)
    
    start = time.perf_counter()
    f_k_dft = dft(f_n)
    dft_times.append(time.perf_counter() - start)
    
    start = time.perf_counter()
    f_k_fft = fft(f_n)
    fft_times.append(time.perf_counter() - start)
    
fig, ax = plt.subplots()

ax.scatter(Ns, dft_times, label="DFT")
ax.scatter(Ns, fft_times, label="FFT")
ax.legend()
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("N")
ax.set_ylabel("computational time (s)")
ax.grid()

plt.show()

We see that the FFT version is much faster for large $N$

# Exercises (10 pt)

## Exercise 1 - Muddiest Moment (2 pt)

What was the most confusing part of this lecture?

**Answer here**


## Exercise 2 - X-ray Burst Timing (8 pt)

The [Rossi X-ray Timing Explorer](https://en.wikipedia.org/wiki/Rossi_X-ray_Timing_Explorer) (RXTE) has very high time resolution and has been used to observe a large number of X-ray sources.  We'll look at the data from the low-mass X-ray binary 4U 1728-34&mdash;this is an [X-ray burst](https://en.wikipedia.org/wiki/X-ray_burster) system.

In [Strohmayer et al. 1996](https://ui.adsabs.harvard.edu/abs/1996ApJ...469L...9S/abstract) it was shown that the neutron star spin rate can be seen in the Fourier transform of the lightcurve of the burst.  Here we repeat the analysis.

### Exercise 2(a) - Downloading and plotting the data (2 pt)

Download the data with the command

```bash
curl -O https://zingale.github.io/computational_astrophysics/_downloads/16f3167db5254e18cb8426c606bf9386/4u1728_burstdata.txt
```

Upload the data as a numpy array `data`, and flatten using

```python
data = np.loadtxt("4u1728_burstdata.txt").flatten()
```

Define a variable `N` as the length of the data array:

```python
N = len(data)
```

Print `N`, and verify it is a power of 2.

Plot the data using the following commands:

```python
fig, ax = plt.subplots()
ax.plot(data)
ax.set_ylabel("counts")
ax.set_xlabel("time bins")
```

### Exercise 2(b) - Bin the data sample (2 pt)

We'll bin the data into fewer samples to build up the signal to noise. We can do this by reshaping it into a 2-d array and then summing over the columns. Do this with the command

```python
binned_data = data.reshape(N // 256, 256).sum(axis=1)
```

We also know that the data is 32 s, so we can put a physical time. Define a time array with

```python
T = 32
times = np.linspace(0, T, len(binned_data), endpoint=False)
```

Plot the data with

```python
fig, ax = plt.subplots()
ax.plot(times, binned_data)
ax.set_xlabel("time (s)")
ax.set_ylabel("counts")
```

### Exercise 2(c) - Take the FFT (2 pt)

Next we want to take the FFT and look to see if there are any interesting frequencies with a lot of power. Do this with numpy with the command

```python
c_k = np.fft.rfft(data)
```

We want to get the physical frequencies corresponding to the Fourier coefficients. The data encompasses 32 s (and with 262144 points, has a sampling rate of 1/8192 s). NumPy's `rfftfreq` can take the sampling rate to return physical frequencies:

```python
rate = T / N
kfreq = np.fft.rfftfreq(N, d=rate)
```

Now plot the power spectrum, and normalize by $2/N$, using

```python
fig, ax = plt.subplots()
ax.plot(kfreq, np.abs(c_k)**2 * 2 / N)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel(r"$\nu_k$")
ax.set_ylabel("power")
```

Here we see a signal around 300 Hz.

### Exercise 2(d) - Further bin and zoom in on relevant frequency. (2 pt)

We'll bin the Fourier data to increase the signal to noise.  The original paper did this by a factor of 8. Do this with the command

```python
c_k_binned = np.abs(c_k[1:]).reshape(int(len(c_k)//8), 8).mean(axis=1)
kfreq_binned = kfreq[1:].reshape(int(len(kfreq)//8), 8).mean(axis=1)
```

Plot the frequencies, and zoom in on the peak around 300 Hz. 

```python
fig, ax = plt.subplots()
ax.plot(kfreq_binned, c_k_binned**2 * 2 / N)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$\nu_k$")
ax.set_ylabel("power")

ax.set_xscale("linear")
ax.set_yscale("linear")

ax.set_xlim(340, 380)
ax.set_ylim(0, 20)

```

We see the strong signal at around 363 Hz&mdash;this is the same frequency identified as the neutron star rotation rate in the original paper.